[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)

# Session 5 Demo: Prompt Engineering with Gemma 4 E2B

This notebook compares two prompt styles on the same course question.

- **zero_shot**: a short direct prompt.
- **structured**: role + task + audience + constraints + output format.

The model answer is requested in English. The output is streamed from the LiteRT-LM CLI.

## 1. Setup

For macOS Apple Silicon, the default configuration below uses the GPU backend and speculative decoding.

Before running the model, verify that LiteRT-LM can start:

```bash
uvx litert-lm run --help
```

The first real run downloads the model from Hugging Face.

In [ ]:
from pathlib import Path
import re
import shlex
import subprocess
import sys

HF_REPO = "litert-community/gemma-4-E2B-it-litert-lm"
MODEL_FILE = "gemma-4-E2B-it.litertlm"
SYLLABUS_PDF = Path("syllabus/GAAI-AIMS-Applied-GenAI-LLMs-Agents_psw.pdf")

# Use "uvx" for a one-shot run. Use "litert-lm" if you installed the CLI permanently.
RUNNER = "uvx"

# On Apple Silicon, try GPU first. If it fails, switch to "cpu".
BACKEND = "gpu"

# Recommended by LiteRT-LM for GPU backends.
ENABLE_SPECULATIVE_DECODING = True

# Requested response limit. This is enforced through the prompt instruction.
MAX_RESPONSE_TOKENS = 220

QUESTION = (
    "How should we adapt a large language model for a low-resource language "
    "such as Wolof in an AIMS Senegal project? Answer in English."
)

## 2. Two Prompt Types

The goal is to show students that the same model can produce different quality and structure depending on the prompt.

In [ ]:
PROMPT_TEMPLATES = {
    "zero_shot": (
        "Explain how to adapt a large language model for a low-resource "
        "language like Wolof. Answer in English. Be concise and explicit. "
        "Use at most {max_response_tokens} tokens. Use plain text only. "
        "Do not use Markdown bold, italic, or heading markers."
    ),
    "structured": (
        "You are an AI engineering instructor at AIMS Senegal.\n\n"
        "Task:\n"
        "Explain how to adapt a large language model for a low-resource "
        "language such as Wolof.\n\n"
        "Audience:\n"
        "Master-level students who know Python and basic machine learning.\n\n"
        "Constraints:\n"
        "- Answer in English.\n"
        "- Prefer practical engineering steps over marketing language.\n"
        "- Distinguish prompting, RAG, LoRA/QLoRA fine-tuning, and evaluation.\n"
        "- Mention data quality, tokenization, safety, cost, and reproducibility.\n"
        "- Do not claim that fine-tuning is always required.\n\n"
        "Formatting rules:\n"
        "- Plain text only.\n"
        "- Do not use Markdown bold or italic markers.\n"
        "- Do not use '**', '*', '__', or markdown heading markers.\n"
        "- Use numbered section titles without bold formatting.\n"
        "- Use '-' for bullet points.\n"
        "- In the French summary, do not write bold labels such as '**Données**'.\n\n"
        "Output format:\n"
        "1. Short answer\n"
        "2. Recommended pipeline\n"
        "3. Prompting vs RAG vs fine-tuning\n"
        "4. Evaluation checklist\n"
        "5. Résumé en français: exactly 3 bullet points\n\n"
        "Length limit:\n"
        "Keep the full answer concise and explicit. Use at most "
        "{max_response_tokens} tokens total.\n\n"
        "Expected style example:\n"
        "1. Short answer\n"
        "Plain sentence without Markdown.\n"
        "5. Résumé en français\n"
        "- Première idée concise.\n"
        "- Deuxième idée concise.\n"
        "- Troisième idée concise."
    ),
}

def build_prompt(name, max_response_tokens=MAX_RESPONSE_TOKENS):
    return PROMPT_TEMPLATES[name].format(max_response_tokens=max_response_tokens)

PROMPTS = {
    "zero_shot": build_prompt("zero_shot"),
    "structured": build_prompt("structured"),
}

print("Course question:\n", QUESTION)
print("\nZero-shot prompt:\n", PROMPTS["zero_shot"])
print("\nStructured prompt:\n", PROMPTS["structured"])

## 3. Streaming Runner

This function calls the LiteRT-LM CLI and streams characters as they arrive.

In [ ]:
def build_command(prompt, runner=RUNNER, backend=BACKEND, enable_mtp=ENABLE_SPECULATIVE_DECODING):
    if runner == "uvx":
        command = ["uvx", "litert-lm", "run"]
    else:
        command = ["litert-lm", "run"]

    command.extend([
        f"--from-huggingface-repo={HF_REPO}",
        MODEL_FILE,
        f"--backend={backend}",
    ])

    if enable_mtp:
        command.append("--enable-speculative-decoding=true")

    command.append(f"--prompt={prompt}")
    return command


def clean_markdown_markers(text):
    for marker in ("*", "_", "#", "`"):
        text = text.replace(marker, "")
    return text


def run_streaming(prompt, clean_markdown=True):
    command = build_command(prompt)
    print("Command:\n" + shlex.join(command) + "\n")
    process = subprocess.Popen(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=0,
    )

    collected = []
    assert process.stdout is not None
    while True:
        chunk = process.stdout.read(1)
        if chunk == "" and process.poll() is not None:
            break
        if chunk:
            if clean_markdown:
                chunk = clean_markdown_markers(chunk)
            collected.append(chunk)
            sys.stdout.write(chunk)
            sys.stdout.flush()

    return process.wait(), "".join(collected).strip()

## 4. Demo A: Zero-Shot Prompt

Use this first to show the baseline behavior.

In [ ]:
run_streaming(PROMPTS["zero_shot"])

## 5. Demo B: Structured Engineering Prompt

Use this second to show how constraints and output format make the answer easier to evaluate and reuse.

In [ ]:
run_streaming(PROMPTS["structured"])

## 6. Chat Mode with Mini-RAG on the Syllabus

This mode turns the model into a GenAI teaching assistant. It retrieves small pieces of context from the local syllabus PDF and uses them when answering student questions.

The retrieval below is intentionally simple: chunk the syllabus text, score chunks by keyword overlap with the student question, then inject the top chunks into the prompt.

In [ ]:
def read_syllabus_text(pdf_path=SYLLABUS_PDF):
    if not pdf_path.exists():
        raise FileNotFoundError(f"Syllabus PDF not found: {pdf_path}")

    cache_path = pdf_path.with_suffix(".txt")
    if cache_path.exists() and cache_path.stat().st_mtime >= pdf_path.stat().st_mtime:
        return cache_path.read_text(encoding="utf-8", errors="ignore")

    from pypdf import PdfReader

    reader = PdfReader(str(pdf_path))
    parts = []
    for page_index, page in enumerate(reader.pages, start=1):
        text = page.extract_text() or ""
        if text.strip():
            parts.append(f"[Syllabus page {page_index}]\n{text}")

    syllabus_text = "\n\n".join(parts).strip()
    cache_path.write_text(syllabus_text, encoding="utf-8")
    return syllabus_text


def chunk_text(text, max_chars=1400, overlap=220):
    clean = re.sub(r"\s+", " ", text).strip()
    chunks = []
    start = 0
    while start < len(clean):
        end = min(start + max_chars, len(clean))
        chunks.append(clean[start:end])
        if end == len(clean):
            break
        start = max(end - overlap, start + 1)
    return chunks


def tokenize_for_retrieval(text):
    stopwords = {"the", "and", "for", "with", "that", "this", "you", "are", "from", "how", "what", "why", "when", "qui", "que", "les", "des", "pour", "dans", "avec", "est", "une", "sur", "cours", "course"}
    terms = re.findall(r"[a-zA-ZÀ-ÿ0-9_+-]{3,}", text.lower())
    return [term for term in terms if term not in stopwords]


def retrieve_context(question, chunks, top_k=3):
    query_terms = set(tokenize_for_retrieval(question))
    if not query_terms:
        return "\n\n---\n\n".join(chunks[:top_k])

    scored = []
    for idx, chunk in enumerate(chunks):
        chunk_terms = set(tokenize_for_retrieval(chunk))
        score = len(query_terms & chunk_terms)
        scored.append((score, idx, chunk))

    scored.sort(key=lambda item: (-item[0], item[1]))
    selected = [chunk for score, _, chunk in scored[:top_k] if score > 0]
    if not selected:
        selected = [chunk for _, _, chunk in scored[:top_k]]
    return "\n\n---\n\n".join(selected)


syllabus_text = read_syllabus_text()
syllabus_chunks = chunk_text(syllabus_text)
print(f"Loaded {len(syllabus_chunks)} syllabus chunks from {SYLLABUS_PDF}")

In [ ]:
def build_chat_prompt(user_question, context, history=None, max_response_tokens=MAX_RESPONSE_TOKENS):
    history = history or []
    recent_history = "\n\n".join(
        f"Student: {u}\nAssistant: {a}" for u, a in history[-3:]
    ) or "No previous turns."

    return (
        "You are a patient GenAI teaching assistant for AIMS Senegal.\n"
        "Your role is to help students understand the current Applied Generative and Agentic AI course.\n\n"
        "Teaching style:\n"
        "- Be concise, explicit, and pedagogical.\n"
        "- Define key terms when useful.\n"
        "- Use one simple analogy when it helps.\n"
        "- Use plain text only: no Markdown bold, no italic markers, no headings with #.\n"
        "- Use the syllabus context if relevant.\n"
        "- If the context is not enough, say what is missing instead of inventing.\n"
        "- Answer in the student's language unless they ask otherwise.\n"
        f"- Keep the answer under {max_response_tokens} tokens.\n\n"
        "Relevant syllabus context:\n"
        f"{context if context.strip() else 'No syllabus context retrieved.'}\n\n"
        "Recent conversation:\n"
        f"{recent_history}\n\n"
        "Student question:\n"
        f"{user_question}\n\n"
        "Assistant answer:"
    )


def ask_course_assistant(question, history=None, top_k=3):
    history = history or []
    context = retrieve_context(question, syllabus_chunks, top_k=top_k)
    prompt = build_chat_prompt(question, context, history)
    exit_code, answer = run_streaming(prompt)
    history.append((question, answer))
    return history


def interactive_chat():
    history = []
    print("Ask questions about the GAAI course. Type 'exit' to stop.")
    while True:
        question = input("Student question: ").strip()
        if question.lower() in {"exit", "quit", "q"}:
            break
        if question:
            history = ask_course_assistant(question, history=history)
            print("\n")
    return history

### Single chat question

Use this for a controlled classroom demo before opening a full interactive loop.

In [ ]:
history = []
history = ask_course_assistant("What is the difference between prompting and RAG?", history)

### Full interactive chat session

Run this cell to let students ask several questions. Type `exit` to stop the chat.

In [ ]:
interactive_chat()

## Teaching Point

Ask students to compare both outputs on four criteria:

- correctness;
- format compliance;
- actionability;
- reproducibility.